# P09 — Prédiction de la disponibilité des stations Vélib'

**Contexte métier.** Vélib' Métropole, opérateur Smovengo et applications d'itinéraires : anticiper les stations pleines ou vides pour le rééquilibrage (camions de redistribution) et guider l'usager vers une station avec vélo/place au moment où il arrive. Enjeu opérationnel : 20,7 % des stations sont quasi vides sur un instantané.

**Objectif de l'EDA.** Construire la cible (taux d'occupation d'une station) et les régresseurs (capacité, commune, proportion d'électriques, borne de paiement).

**Données.** Vélib' — disponibilité en temps réel — Ville de Paris (opendata.paris.fr). Licence ouverte Etalab 2.0. Instantané de 1 519 stations (l'API est un flux : chaque appel est une photo à l'instant T).

## 1. Téléchargement

Instantané du flux temps réel via l'API opendata.paris.fr :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/velib-disponibilite-en-temps-reel/exports/csv"
if not os.path.exists("data/velib-disponibilite.csv"):
    urllib.request.urlretrieve(URL, "data/velib-disponibilite.csv")
print(os.path.getsize("data/velib-disponibilite.csv") / 1e6, "Mo")

## 2. Chargement

Séparateur `;`. Le taux d'occupation = vélos disponibles / capacité ; le taux d'électriques = électriques / vélos.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/velib-disponibilite.csv", sep=";", low_memory=False, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
df["cap"] = pd.to_numeric(df["Capacité de la station"], errors="coerce")
df["velos"] = pd.to_numeric(df["Nombre total vélos disponibles"], errors="coerce")
df["elec"] = pd.to_numeric(df["Vélos électriques disponibles"], errors="coerce")
df["libres"] = pd.to_numeric(df["Nombre bornettes libres"], errors="coerce")
df["taux_occ"] = df["velos"] / df["cap"] * 100
df["taux_elec"] = df["elec"] / df["velos"].replace(0, np.nan) * 100
print(len(df), "stations")

## 3. Analyse

Distribution du taux d'occupation, stations vides/pleines, variation par commune, part du vélo électrique et corrélation capacité/disponibilité.

In [ ]:
print("Taux d'occupation (%) :", df["taux_occ"].describe(percentiles=[.1, .5, .9]).round(1).to_dict())
print("Stations quasi vides (<10 %) :", round((df["taux_occ"] < 10).mean() * 100, 1), "%")
print("Stations pleines (0 bornette libre) :", round((df["libres"] == 0).mean() * 100, 1), "%")
print()
g = df.groupby("Nom communes équipées")["taux_occ"].agg(["median", "count"])
g = g[g["count"] >= 5].sort_values("median")
print("Communes les moins remplies :", g.head(5).round(1).to_dict("index"))
print("Communes les plus remplies :", g.tail(5).round(1).to_dict("index"))
print()
print("Médiane taux d'électriques :", round(df["taux_elec"].median(), 1), "%")
print("Corr. Spearman capacité vs vélos dispo :", round(df[["cap", "velos"]].corr("spearman").iloc[0, 1], 2))

## 4. Résultats observés (exécution du 16/10/2026)

- n stations : 1519
- en fonctionnement : OUI : 1517, NON : 2
- capacite stats : count : 1 519.00, mean : 32.80, std : 18.90, min : 0.00, 10% : 20.00, 25% : 23.00, 50% : 30.00, 75% : 38.50, 90% : 50…
- taux occupation stats : count : 1 516.00, mean : 35.50, std : 28.90, min : 0.00, 10% : 5.00, 25% : 11.40, 50% : 26.40, 75% : 56.30, 90% : 83.…
- pct stations presque vides : 20.70
- pct stations pleines : 3.10
- part velo electrique mediane : 50.00
- capacite par commune : Paris : 34715, Boulogne-Billancourt : 908, Montreuil : 754, Issy-les-Moulineaux : 728, Saint-Denis : 666, Ivry-sur-Se…
- borne paiement : OUI : 1502, NON : 17
- velos dispo stats : count : 1 519.00, mean : 11.60, std : 11.60, min : 0.00, 25% : 3.00, 50% : 8.00, 75% : 16.00, max : 78.00
- corr spearman cap velos : 0.31
- actualisation : 2021-02-21T19:09:18+00:00 → 2026-10-16T15:53:24+00:00

## 5. Variable cible et régresseurs

**Cible (régression) : `taux_occupation` de la station, ou variantes binaires "station bientôt vide" / "station pleine".** Au niveau métier, c'est une prévision à 15-60 min (streaming), pas une classification statique.

**Régresseurs retenus :**
- `Capacité de la station` (corr. Spearman 0,42 avec vélos disponibles) et coordonnées géographiques
- `Nom communes équipées` : médiane d'occupation de 9,4 % (Bagnolet) à 81,7 % (Bois-Colombes) — très discriminant
- Proportion d'électriques (taux médian 66,7 %) : les électriques influencent les flux (bornes de charge)
- Heure/jour (si l'on archive plusieurs instantanés — voir pièges)

**Pièges :**
- Un seul instantané = pas de dimension temporelle : pour un vrai modèle prédictif, il faut archiver le flux toutes les 5-15 min (API opendata ou champ `Actualisation de la donnée`) et apprendre la trajectoire des stations
- Ne pas prédire la disponibilité à l'instant T avec des features du même instant : la cible doit être l'état futur (T+30 min)
- Deux stations hors fonctionnement ; capacité 0 pour certaines : nettoyer
- Les stations en jumeaux (capacité 0) et doublons INSEE sont à dédupliquer